In [1]:
# ============================================================
#  HyPA-RAG — Notebook Unificado
#  Reindexação incremental de PDFs + Servidor Flask
# ============================================================

from __future__ import annotations

import os, re, json, time, hashlib, threading, webbrowser
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from datetime import datetime

import numpy as np
import redis
import requests
from requests.exceptions import Timeout, ConnectionError as ReqConnectionError, HTTPError
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from flask import Flask, request, render_template_string, jsonify

from redis.commands.search.field import TextField, VectorField, TagField, NumericField
from redis.commands.search.query import Query
from redis.commands.search.index_definition import IndexDefinition, IndexType

c:\Users\adede\OneDrive\Documents\hyparag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================
#  HyPA-RAG — Configuração principal
# ============================================================

# ── 0. Configuração dual-model ───────────────────────────────
MODEL_NAME     = "meta-llama-3.1-8b-instruct"                    # gerador
JUDGE_PROVIDER = "local"
JUDGE_API_URL  = "http://localhost:1234/v1/chat/completions"  # LM Studio
JUDGE_MODEL    = "qwen2.5-7b-instruct"
THRESHOLD_RECALL = 0.834

# ── Reranker ────────────────────────────────────────────────
RERANKER_MODEL_NAME  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANKER_DEVICE      = "cpu"          # troque para "cuda" se tiver GPU
RERANKER_BATCH_SIZE  = 8
USE_RERANKER         = True           # False desativa sem alterar código

# ── LM Studio ───────────────────────────────────────────────
LM_API_URL = os.getenv("LM_API_URL", "http://localhost:1234/v1/chat/completions")
HEADERS    = {"Content-Type": "application/json"}
MODEL_NAME = os.getenv("MODEL_NAME", MODEL_NAME)

# ── Redis ────────────────────────────────────────────────────
REDIS_HOST     = os.getenv("REDIS_HOST",     "localhost")
REDIS_PORT     = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", None)
REDIS_DB       = int(os.getenv("REDIS_DB",   "0"))
REDIS_INDEX    = os.getenv("REDIS_INDEX",    "idx:hyparag_v13")
REDIS_PREFIX   = os.getenv("REDIS_PREFIX",   "doc:v13:")

# ── Áreas temáticas (v13) ────────────────────────────────────
AREAS = [
    "organizacao_administrativa",
    "processo_administrativo",
    "servidor_publico",
    "educacao",
    "licitacoes_contratos",
    "financas_publicas",
    "transparencia_controle",
    "instrumentos_planejamento",
    "norma_geral",
    "outros",
]

# ── Embeddings ───────────────────────────────────────────────
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "intfloat/e5-base-v2")
print("Carregando modelo de embeddings…")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
EMBED_DIM   = embedding_model.get_sentence_embedding_dimension()
EMBED_DTYPE = np.float32

def encode_queries(texts: list) -> np.ndarray:
    prefixed = [f"query: {t}" for t in texts]
    return embedding_model.encode(prefixed, normalize_embeddings=True)

def encode_passages(texts: list) -> np.ndarray:
    prefixed = [f"passage: {t}" for t in texts]
    return embedding_model.encode(prefixed, normalize_embeddings=True)

# ── Métricas / logging ───────────────────────────────────────
RAG_EVAL_LOG       = os.getenv("RAG_EVAL_LOG",    "rag_metrics_log.jsonl")
RAG_EVAL_ENABLE    = os.getenv("RAG_EVAL_ENABLE", "1") == "1"
RAG_EVAL_LLM_MODEL = os.getenv("RAG_EVAL_LLM_MODEL", MODEL_NAME)

print("OK: Config carregada")


In [ ]:
# ── 2. Helpers básicos ───────────────────────────────────────
def as_text(v):
    if v is None:
        return ""
    if isinstance(v, (bytes, bytearray)):
        return v.decode("utf-8", errors="ignore")
    return str(v)

def get_redis_client():
    return redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        password=REDIS_PASSWORD,
        db=REDIS_DB,
        decode_responses=False
    )

r = get_redis_client()

print("OK: Helpers/Redis prontos")

# ── 3b. Classificação de ÁREA (v13) ──────────────────────────
# Regras simples por palavras‑chave no texto + fallback por nome do arquivo.
# Objetivo: popular o campo TAG "area" para permitir filtro no retrieve.

AREA_RULES = [
    ("instrumentos_planejamento", [
        "plano plurianual", "ppa", "ldo", "lei de diretrizes orçamentárias",
        "loa", "lei orçamentária anual", "planejamento", "metas e prioridades",
        "programa de governo", "planejamento estratégico",
    ]),
    ("transparencia_controle", [
        "transparência", "acesso à informação", "lai", "ouvidoria", "sic",
        "prestação de contas", "controle interno", "controle social",
        "conselho fiscal", "auditoria", "corregedoria",
    ]),
    ("licitacoes_contratos", [
        "licitação", "pregão", "concorrência", "dispensa", "inexigibilidade",
        "contrato", "termo aditivo", "ata de registro de preços",
    ]),
    ("financas_publicas", [
        "receita", "despesa", "empenho", "liquidação", "pagamento",
        "orçamento", "tesouraria", "contabilidade", "lrf", "responsabilidade fiscal",
        "crédito adicional", "suplementar", "balanço", "dívida",
    ]),
    ("processo_administrativo", [
        "processo administrativo", "procedimento", "autuação", "trâmite",
        "prazos", "recurso", "petição", "intimação", "notificação",
        "processo eletrônico", "pae",
    ]),
    ("servidor_publico", [
        "servidor", "cargo", "carreira", "remuneração", "gratificação",
        "licença", "afastamento", "aposentadoria", "pensão", "concurso público",
        "estágio probatório", "regime jurídico", "vencimento",
    ]),
    ("organizacao_administrativa", [
        "estrutura organizacional", "competência", "atribuições", "organograma",
        "secretaria", "autarquia", "fundação", "órgão", "entidade",
        "regimento", "decreto de organização", "unidade administrativa",
    ]),
    ("educacao", [
        "educação", "escola", "aluno", "professor", "ensino", "currículo",
        "matrícula", "rede estadual", "rede municipal", "universidade",
    ]),
    ("norma_geral", [
        "disposições gerais", "normas gerais", "revogam-se", "vigência",
        "regulamenta", "aplica-se", "interpretação",
    ]),
]

def _norm(s: str) -> str:
    s = (s or "").lower()
    # normalização bem simples, sem depender de libs extras
    s = s.replace("á","a").replace("à","a").replace("ã","a").replace("â","a")
    s = s.replace("é","e").replace("ê","e")
    s = s.replace("í","i")
    s = s.replace("ó","o").replace("ô","o").replace("õ","o")
    s = s.replace("ú","u")
    s = s.replace("ç","c")
    return s

def inferir_area(texto: str, source: str = "") -> str:
    t = _norm(texto)
    s = _norm(source)
    # dica por nome do arquivo (ex.: "..._lai.pdf")
    if "lai" in s:
        return "transparencia_controle"
    for area, kws in AREA_RULES:
        for kw in kws:
            if _norm(kw) in t:
                return area
    return "outros"

# Mapeamento de termos em linguagem natural → área
# Ordem importa: mais específico primeiro

_AREA_KW_PERGUNTA: list[tuple[str, list[str]]] = [
    ("transparencia_controle", [
        "transparencia", "transparência", "acesso a informacao", "acesso a informação",
        "acesso à informação", "lei de acesso", "lai", "ouvidoria", "sic", "e-sic",
        "pedido de acesso a informação", "pedido de acesso à informação", "direito de acesso à informação",
        "prestacao de contas", "portal da transparencia",
        "portal transparencia", "controle social", "controle externo",
        "dados abertos", "publicidade dos atos", "sigilo", "informação sigilosa",
    ]),
    ("licitacoes_contratos", [
        "licitacao", "licitação", "pregao", "pregão", "contrato administrativo",
        "dispensa", "inexigibilidade", "ata de registro", "edital",
        "termo de referencia", "comissao de licitacao",
    ]),
    ("financas_publicas", [
        "orcamento", "orçamento", "receita publica", "despesa publica",
        "empenho", "liquidacao", "responsabilidade fiscal", "lrf",
        "tesouraria", "contabilidade publica", "credito adicional",
    ]),
    ("instrumentos_planejamento", [
        "ppa", "plano plurianual", "ldo", "lei de diretrizes orcamentarias",
        "loa", "lei orcamentaria", 
        "metas e prioridades", "programa de governo",
    ]),
    ("processo_administrativo", [
        "processo administrativo", "procedimento administrativo",
        "prazo processual", "recurso administrativo", "intimacao",
        "processo eletronico", "pae", "sei", "autuacao",
    ]),
    ("servidor_publico", [
        "servidor publico", "cargo publico", "concurso", "concurso publico",  # ← ambos
        "remuneracao", "gratificacao", "licenca", "afastamento",
        "aposentadoria", "regime juridico", "progressao", "ferias",
    ]),
    ("organizacao_administrativa", [
        "estrutura organizacional", "competencia", "atribuicoes",
        "secretaria de estado", "autarquia", "fundacao", "regimento interno",
        "decreto de organizacao", "unidade administrativa",
    ]),
    ("educacao", [
        "educacao", "escola", "ensino", "aluno", "matricula",
        "docente", "universidade", "curriculo", "rede estadual",
    ]),
    ("norma_geral", [
        "constituicao", "codigo", "lei federal", "decreto-lei",
        "disposicoes gerais", "normas gerais",
    ]),
]

# Optional, Tuple já importados na cell 1
# ============================================================
# HELPERS DE FILTRO REDIS (V13)
# ============================================================

def escape_tag_value(v: str) -> str:
    """
    Escapa caracteres especiais para TAG do RediSearch.
    ESSENCIAL para nomes de arquivos com '.' ou '-'.
    """
    return re.sub(r"([^a-zA-Z0-9_])", r"\\\1", v)


# detectar_source_na_pergunta definida na cell 7  — versão canônica

def detectar_area_na_pergunta(pergunta: str) -> Tuple[Optional[str], float, str]:
    p = _norm(pergunta)

    # 1) slug explícito ("area: transparencia_controle") -> altíssima confiança
    m = re.search(r'\barea\s*[:=]\s*([a-z_]+)', p)
    if m and m.group(1) in AREAS:
        return m.group(1), 1.0, "explicit_area_param"

    # 2) slug literal na pergunta -> alta confiança
    for a in AREAS:
        if a in p:
            return a, 0.95, "literal_slug"

    # 3) keywords -> confiança variável (conta quantos hits)
    best_area = None
    best_hits = 0
    best_kw = None

    for area, kws in _AREA_KW_PERGUNTA:
        hits = 0
        hit_kw = None
        for kw in kws:  # ← fix: loop que estava faltando
            if _norm(kw) in p:
                hits += 1
                if hit_kw is None:
                    hit_kw = kw
        if hits > best_hits:
            best_hits = hits
            best_area = area
            best_kw = hit_kw

    if best_area and best_hits > 0:
        # heurística simples:
        # 1 hit -> fraco; 2+ hits -> moderado/forte
        conf = 0.55 if best_hits == 1 else 0.75
        reason = f"keyword_hits={best_hits} first_kw={best_kw}"
        return best_area, conf, reason

    return None, 0.0, "no_match"


# ── 5. Retrieval HyPA-RAG ────────────────────────────────────
_PADROES_COMPLEXIDADE = {
    "alta": [
        r"\bcompar[ae]\b", r"\bdiferença\b", r"\bversus\b", r"\bvs\.?\b",
        r"\bcontrast[ae]\b", r"\bentre.*e\b",
        r"\bart\.?\s*\d+.*e.*art\.?\s*\d+",
        r"\bquais (são|seriam) os\b",
        r"\bquando.*e.*quando\b",
        r"\bcomo.*e.*como\b",
        r"\bpor que.*e\b",
        r"\be\s+(?:também|ainda|além)\b",
        r"\bexplique\b", r"\bdiscorra\b", r"\banalise\b",
        r"\brelação entre\b", r"\bcritérios para\b",
    ],
    "media": [
        r"\bquando\b", r"\bonde\b", r"\bcomo\b",
        r"\bpor que\b", r"\bquais\b",
        r"\bresponsabilidade\b", r"\bprocedimento\b",
        r"\bprazo\b", r"\bcondição\b",
    ],
}

def classificar_complexidade(pergunta: str) -> str:
    p = pergunta.strip().lower()
    n = len(p)
    n_artigos = len(re.findall(r"art\.?\s*\d+", p))
    for padrao in _PADROES_COMPLEXIDADE["alta"]:
        if re.search(padrao, p):
            return "complexa"
    if n_artigos >= 2:
        return "complexa"
    for padrao in _PADROES_COMPLEXIDADE["media"]:
        if re.search(padrao, p):
            return "media" if n < 180 else "complexa"
    if n < 70:
        return "simples"
    if n < 160:
        return "media"
    return "complexa"

def params_por_complexidade(nivel: str) -> dict:
    tabela = {
        #           k_dense  k_sparse  k_final
        "simples":  {"k_dense": 12, "k_sparse":  8, "k_final": 2},
        "media":    {"k_dense": 20, "k_sparse": 12, "k_final": 3},
        "complexa": {"k_dense": 30, "k_sparse": 15, "k_final": 3},
    }
    return tabela[nivel]

def escape_redisearch_query(text: str) -> str:
    if not text:
        return "*"
    text = text.replace('"', " ").replace("'", " ")
    text = re.sub(r"""[()\[\]{}:@~*^|=<>\\/?!$%&;,\.]""", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "*"

def extrair_termos_chave(pergunta: str, max_termos: int = 5) -> str:
    """
    Extrai substantivos/termos relevantes da query para o BM25.
    Remove stopwords comuns do português para evitar AND restritivo.
    """
    STOPWORDS = {
        "qual", "quais", "como", "quando", "onde", "por", "que",
        "de", "do", "da", "dos", "das", "em", "no", "na",
        "nos", "nas", "o", "a", "os", "as", "um", "uma", "uns",
        "umas", "e", "ou", "se", "para", "com", "sem", "sob",
        "sobre", "entre", "ate", "é", "são", "sao", "foi", "sera",
        "pode", "deve", "me", "explique", "explica", "fale",
        "diga", "descreva", "liste", "quero", "saber",
        "este", "esta", "esse", "essa", "estes", "estas", "esses", "essas",
        "seria", "seriam", "estão", "estao", "tem", "têm", "tем",
        "há", "ha", "ter", "ser",
    }
    palavras = re.sub(r"[^\w\s]", " ", pergunta.lower()).split()
    termos   = [p for p in palavras if p not in STOPWORDS and len(p) > 2]
    # Expansão morfológica simples: adiciona variações de termos-chave
    _expansoes = {
        "decretos":        ["decreto", "decretos"],
        "decreto":         ["decreto", "decretos"],
        "leis":            ["lei", "leis"],
        "transparencia":   ["transparência", "transparencia"],
        "transparência":   ["transparência", "transparencia"],
        "administrativo":  ["administrativo", "administrativa", "administrativos"],
        "eletronico":      ["eletrônico", "eletronico", "eletrônicos"],
        "eletrônico":      ["eletrônico", "eletronico", "eletrônicos"],
    }
    expandidos = []
    for t in termos[:max_termos]:
        if t.lower() in _expansoes:
            expandidos.extend(_expansoes[t.lower()])
        else:
            expandidos.append(t)
    # Remove duplicatas mantendo ordem
    vistos = set()
    termos_final = []
    for t in expandidos:
        if t not in vistos:
            vistos.add(t)
            termos_final.append(t)
    return " | ".join(termos_final)   # OR entre os termos


def rrf_fusion(listas, k_rrf=60, pesos=None):
    """
    Reciprocal Rank Fusion com pesos por lista.

    listas : [dense_list, sparse_list]
    pesos  : [2.0, 1.0]  → dense tem o dobro do peso
    """
    if pesos is None:
        pesos = [1.0] * len(listas)

    scores, payload = {}, {}

    for lst, w in zip(listas, pesos):
        for rank, item in enumerate(lst, start=1):
            docid = item["redis_id"]
            scores[docid] = scores.get(docid, 0.0) + w * (1.0 / (k_rrf + rank))
            if docid not in payload:
                payload[docid] = item
            # propaga rrf_score para uso no selector
            payload[docid]["rrf_score"] = scores[docid]

    # propaga scores finais
    for docid, sc in scores.items():
        payload[docid]["rrf_score"] = sc

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [payload[docid] for docid, _ in ranked]

def selecionar_contexto_final(
    candidatos,
    k_final,
    pergunta="",
    max_chars=9000,
    max_chunks_por_doc=2,
    min_rrf_score=0.0,  # ✅ PATCH: aceita o parâmetro usado na chamada
):
    if not candidatos:
        return []

    if USE_RERANKER and pergunta:
        reranker = get_reranker()
        pares    = [(pergunta, it.get("content", "")) for it in candidatos]
        scores   = reranker.predict(pares, batch_size=RERANKER_BATCH_SIZE)
        for it, sc in zip(candidatos, scores):
            it["reranker_score"] = float(sc)
        # Descarta apenas chunks muito irrelevantes (score < -4 no cross-encoder)
        # Logits do ms-marco: >0 = relevante, 0 a -4 = incerto, <-4 = irrelevante
        candidatos = [c for c in candidatos if c.get("reranker_score", 0) > -4.0]
        candidatos = sorted(candidatos, key=lambda x: x["reranker_score"], reverse=True)

    selecionados   = []
    chunks_por_doc = {}
    total_chars    = 0
    # ← fingerprint para deduplicar por conteúdo
    conteudos_vistos = set()

    for it in candidatos:
        # ✅ PATCH: filtro por score mínimo do RRF
        rrf_score = float(it.get("rrf_score", 0.0) or 0.0)
        if rrf_score < min_rrf_score:
            continue

        doc   = it["source"]
        n_doc = chunks_por_doc.get(doc, 0)
        if n_doc >= max_chunks_por_doc:
            continue

        # ← DEDUPLICAÇÃO: primeiros 120 chars como fingerprint
        fingerprint = it.get("content", "")[:120].strip()
        if fingerprint in conteudos_vistos:
            continue
        conteudos_vistos.add(fingerprint)

        bloco   = f"[Fonte: {it['source']} | Chunk: {it['chunk_id']}]\n{it['content']}"
        tamanho = len(bloco)
        if total_chars + tamanho > max_chars:
            break

        selecionados.append(it)
        chunks_por_doc[doc] = n_doc + 1
        total_chars += tamanho
        if len(selecionados) >= k_final:
            break

    return selecionados

# garante que o reranker não quebre quando ativado
_cross_encoder = None

def get_reranker():
    return _get_cross_encoder()

def _get_cross_encoder():
    global _cross_encoder
    if _cross_encoder is None:
        from sentence_transformers import CrossEncoder
        print("Carregando cross-encoder...")
        _cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        print("Cross-encoder pronto.")
    return _cross_encoder


# Threshold semântico por nível de complexidade:
# - simples/complexa ampla → threshold baixo (query genérica, precisa de mais contexto)
# - media/específica       → threshold alto (query precisa, filtra ruído)
_SCORE_THRESHOLD_BY_NIVEL = {
    "simples":  0.55,
    "media":    0.62,
    "complexa": 0.58,   # complexa costuma ser ampla — threshold menor
}

# ── 6. LLM — geração ─────────────────────────────────────────

def _truncate_text(s: str, max_chars: int = 18000) -> str:
    """Trunca por caracteres para evitar estourar contexto (simples e eficaz)."""
    s = s or ""
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + "\n\n[... CONTEXTO TRUNCADO ...]"

def responder_com_contexto(pergunta: str, area: str | None = None, source: str | None = None):
    contexto, meta, fused = buscar_contexto_hypa_filtrado(pergunta, area=area, source=source)
    contexto = _truncate_text(contexto, max_chars=18000)

    import re as _re

    def _limpar_artefatos(txt: str) -> str:
        txt = (txt or "").strip()
        txt = _re.sub(r"(?im)^\s*###\s*(response|explanation|answer)\s*:?\s*", "", txt).strip()
        txt = _re.sub(r"(?im)^\s*(response|explanation|answer)\s*:?\s*", "", txt).strip()
        return txt

    def _parece_ingles(txt: str) -> bool:
        t = (txt or "").lower()
        t = _re.sub(r"\[fonte:.*?\]", " ", t, flags=_re.IGNORECASE)
        # gatilhos fortes
        if any(x in t for x in ["based on the provided context", "here's", "explanation:", "### explanation", "### response"]):
            return True
        en = sum(t.count(w) for w in [" the ", " and ", " of ", " to ", " in ", " for ", " with ", " is ", " are "])
        pt = sum(t.count(w) for w in [" o ", " a ", " os ", " as ", " de ", " do ", " da ", " e ", " para ", " com ", " que ", " não "])
        return (len(t) > 120) and (en >= 3) and (en > pt)

    prompt = f"""Você é um assistente especializado em análise de documentos jurídicos e normativos do Brasil.

REGRAS OBRIGATÓRIAS
1) Responda SOMENTE em Português do Brasil (pt-BR). Não use inglês ou espanhol.
2) Use APENAS o CONTEXTO. Não use conhecimento externo. Não invente.
3) Entregue apenas a resposta final (sem "Explanation", "Reasoning", "Response:", "###").
4) Se o CONTEXTO estiver vazio ou totalmente irrelevante, responda LITERALMENTE:
Não encontrei essa informação nos documentos disponíveis.
5) Cada frase factual deve terminar com: [Fonte: ... | Chunk: ...]

CONTEXTO
{contexto}

PERGUNTA
{pergunta}

RESPOSTA (somente pt-BR, objetiva):
"""

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Idioma obrigatório: Português do Brasil (pt-BR). "
                    "Proibido responder em outro idioma. "
                    "Use apenas o contexto fornecido. Não invente. "
                    "Sem explicações/cabeçalhos. "
                    "Cada frase factual deve conter [Fonte: ... | Chunk: ...]. "
                    "Se contexto vazio/irrelevante, responda exatamente a frase de fallback."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.1,
        "max_tokens": 1000,
        "top_p": 0.8,
    }

    try:
        resp = requests.post(LM_API_URL, headers=HEADERS, json=payload, timeout=300)
        resp.raise_for_status()
        data = resp.json()

        if "choices" not in data or not data["choices"]:
            if "error" in data:
                raise RuntimeError(f"LM Studio retornou erro: {data['error']}")
            raise RuntimeError(f"Resposta inesperada do servidor: {data}")

        answer = _limpar_artefatos(data["choices"][0]["message"]["content"])

        # Retry único se escapar idioma/artefatos
        if _parece_ingles(answer):
            retry_instruction = (
                "A resposta anterior violou as regras. "
                "Reescreva DO ZERO somente em pt-BR, sem headings, "
                "usando apenas o CONTEXTO e citando fontes ao fim de cada frase factual."
            )
            payload_retry = dict(payload)
            payload_retry["messages"] = [
                payload["messages"][0],
                {"role": "user", "content": f"{retry_instruction}\n\n{prompt}"},
            ]
            resp2 = requests.post(LM_API_URL, headers=HEADERS, json=payload_retry, timeout=300)
            resp2.raise_for_status()
            data2 = resp2.json()
            answer = _limpar_artefatos(data2["choices"][0]["message"]["content"])

        if not answer:
            answer = "Não encontrei essa informação nos documentos disponíveis."

        return answer, meta, fused, contexto

    except (Timeout, ReqConnectionError) as e:
        msg = (
            "Falha de conexão com o LM Studio (timeout/conexão). "
            "Verifique se o servidor local está ativo e se LM_API_URL está correto."
        )
        raise RuntimeError(msg) from e

    except HTTPError as e:
        detail = getattr(resp, "text", "")
        raise RuntimeError(f"HTTP {resp.status_code} ao chamar LM Studio. Detalhe: {detail[:500]}") from e


# ── 7. Métricas de avaliação ─────────────────────────────────
def _cosine(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return float(np.dot(a, b) / ((np.linalg.norm(a) + 1e-12) * (np.linalg.norm(b) + 1e-12)))

def answer_relevance_score(question: str, answer: str) -> float:
    qe = encode_queries([question])[0]
    ae = encode_passages([answer])[0]
    return _cosine(qe, ae)

def _llm_chat(messages, temperature=0.0, max_tokens=512):
    if JUDGE_PROVIDER == "anthropic":
        import anthropic
        client = anthropic.Anthropic(api_key=os.getenv("JUDGE_API_KEY"))
        system_msg = next((m["content"] for m in messages if m["role"] == "system"), "")
        user_msgs  = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=JUDGE_MODEL, max_tokens=max_tokens,
            system=system_msg, messages=user_msgs,
        )
        return resp.content[0].text
    elif JUDGE_PROVIDER == "openai":
        import openai
        client = openai.OpenAI(api_key=os.getenv("JUDGE_API_KEY"))
        resp = client.chat.completions.create(
            model=JUDGE_MODEL, messages=messages,
            temperature=temperature, max_tokens=max_tokens,
        )
        return resp.choices[0].message.content
    else:  # local — LM Studio
        import time
        time.sleep(1.5)  # breve pausa para evitar sobrecarga em chamadas rápidas
        payload = {
            "model": JUDGE_MODEL,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": -1,      # ← era max_tokens, LM Studio aceita -1
            "stream": False,       # ← obrigatório
            "top_p": 0.9,
        }
        resp = requests.post(LM_API_URL, headers=HEADERS, json=payload, timeout=300)
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"]

def _safe_json_extract(text: str):
    text = text.strip()
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def llm_judge_chunk_relevance(question: str, chunk_text: str) -> float:
    messages = [
        {"role": "system", "content": "Você é um avaliador rigoroso de recuperação (RAG). Responda APENAS em JSON."},
        {"role": "user",   "content": f'Retorne JSON: {{"relevance": <0..1>}}\nPERGUNTA: {question}\nCHUNK: {chunk_text}'},
    ]
    out = _llm_chat(messages, temperature=0.0, max_tokens=512)
    obj = _safe_json_extract(out) or {}
    try:
        return max(0.0, min(1.0, float(obj.get("relevance", 0.0))))
    except Exception:
        return 0.0

def context_precision(question: str, retrieved_chunks: list, threshold: float = 0.5):
    if not retrieved_chunks:
        return 0.0, []
    rels = []
    for ch in retrieved_chunks:
        score = llm_judge_chunk_relevance(question, ch.get("content", ""))
        rels.append({"chunk_id": ch.get("chunk_id", ""), "source": ch.get("source", ""), "relevance": score})
    hits = sum(1 for r_ in rels if r_["relevance"] >= threshold)
    return hits / max(1, len(rels)), rels

def llm_judge_faithfulness(question: str, answer: str, context_text: str) -> float:
    messages = [
        {"role": "system", "content": "Avalie groundedness. Responda APENAS em JSON."},
        {"role": "user",   "content": f'Retorne JSON: {{"faithfulness": <0..1>}}\nPERGUNTA: {question}\nCONTEXTO: {context_text}\nRESPOSTA: {answer}'},
    ]
    out = _llm_chat(messages, temperature=0.0, max_tokens=512)
    obj = _safe_json_extract(out) or {}
    try:
        return max(0.0, min(1.0, float(obj.get("faithfulness", 0.0))))
    except Exception:
        return 0.0

def llm_judge_answer_correctness(question: str, answer: str, ground_truth_answer: str) -> float:
    messages = [
        {"role": "system", "content": "Avalie corretude vs GT. Responda APENAS em JSON."},
        {"role": "user",   "content": f'Retorne JSON: {{"correctness": <0..1>}}\nPERGUNTA: {question}\nRESPOSTA: {answer}\nGROUND_TRUTH: {ground_truth_answer}'},
    ]
    out = _llm_chat(messages, temperature=0.0, max_tokens=512)
    obj = _safe_json_extract(out) or {}
    try:
        return max(0.0, min(1.0, float(obj.get("correctness", 0.0))))
    except Exception:
        return 0.0

def context_recall(retrieved_chunks: list, ground_truth_contexts: list) -> float:
    if not ground_truth_contexts:
        return None
    if not retrieved_chunks:
        return 0.0
    gt_embs  = embedding_model.encode(ground_truth_contexts)
    ret_texts = [c.get("content", "") for c in retrieved_chunks]
    ret_embs  = embedding_model.encode(ret_texts) if ret_texts else []
    covered = 0
    for g in gt_embs:
        best = max((_cosine(g, re_) for re_ in ret_embs), default=0.0)
        if best >= THRESHOLD_RECALL:
            covered += 1
    return covered / max(1, len(ground_truth_contexts))

def hit_rate_at_k(retrieved_chunks: list, ground_truth_contexts: list, threshold: float = 0.78) -> float:
    if not ground_truth_contexts or not retrieved_chunks:
        return 0.0
    gt_embs  = encode_passages(ground_truth_contexts)
    ret_embs = encode_passages([c.get("content", "") for c in retrieved_chunks])
    for g in gt_embs:
        for re_ in ret_embs:
            if _cosine(g, re_) >= threshold:
                return 1.0
    return 0.0

def mrr_at_k(retrieved_chunks: list, ground_truth_contexts: list, threshold: float = 0.78) -> float:
    if not ground_truth_contexts or not retrieved_chunks:
        return 0.0
    gt_embs  = encode_passages(ground_truth_contexts)
    ret_embs = encode_passages([c.get("content", "") for c in retrieved_chunks])
    for pos, r_emb in enumerate(ret_embs, start=1):
        for g_emb in gt_embs:
            if _cosine(g_emb, r_emb) >= threshold:
                return 1.0 / pos
    return 0.0

def ndcg_at_k(retrieved_chunks: list, ground_truth_contexts: list, k: int = None, threshold: float = 0.78) -> float:
    if not ground_truth_contexts or not retrieved_chunks:
        return 0.0
    k = k or len(retrieved_chunks)
    gt_embs  = encode_passages(ground_truth_contexts)
    ret_embs = encode_passages([c.get("content", "") for c in retrieved_chunks[:k]])
    rels = [1.0 if max(_cosine(g, r_e) for g in gt_embs) >= threshold else 0.0 for r_e in ret_embs]
    dcg  = sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(ground_truth_contexts), k)))
    return round(dcg / idcg, 4) if idcg > 0 else 0.0

def coletar_metricas_rag(question, answer, retrieved_chunks, context_text,
                          ground_truth_answer=None, ground_truth_contexts=None):
    cp, per_chunk = context_precision(question, retrieved_chunks, threshold=0.25)
    cr    = context_recall(retrieved_chunks, ground_truth_contexts or [])
    faith = llm_judge_faithfulness(question, answer, context_text)
    ar    = answer_relevance_score(question, answer)
    gt_c  = ground_truth_contexts or []
    hr   = hit_rate_at_k(retrieved_chunks, gt_c) if gt_c else None
    mrr  = mrr_at_k(retrieved_chunks, gt_c)      if gt_c else None
    ndcg = ndcg_at_k(retrieved_chunks, gt_c)     if gt_c else None

    metrics = {
        "hit_rate": None if hr   is None else round(hr,   4),
        "mrr":      None if mrr  is None else round(mrr,  4),
        "ndcg":     None if ndcg is None else round(ndcg, 4),
        "context_precision":         round(cp,    4),
        "context_precision_details": per_chunk,
        "context_recall":            None if cr is None else round(cr, 4),
        "faithfulness":              round(faith, 4),
        "answer_relevance":          round(ar,    4),
        "answer_correctness":        None,
    }
    if ground_truth_answer:
        ac = llm_judge_answer_correctness(question, answer, ground_truth_answer)
        metrics["answer_correctness"] = round(ac, 4)
    return metrics

def log_metricas(entry: dict):
    if not RAG_EVAL_ENABLE:
        return
    with open(RAG_EVAL_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("OK: Métricas RAG prontas")

# ── 8. Avaliação de dataset ──────────────────────────────────
def avaliar_dataset(eval_path: str = "eval_set_auto.jsonl", out_path: str = "eval_results.jsonl"):
    if not os.path.exists(eval_path):
        raise FileNotFoundError(f"Arquivo não encontrado: {eval_path}")
    with open(eval_path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    with open(out_path, "w", encoding="utf-8") as fo:
        for row in rows:
            qid      = row.get("id", "")
            question = row.get("question", "")
            gt_a     = row.get("ground_truth_answer", None)
            gt_c     = row.get("ground_truth_contexts", None)
            answer, meta, fused, context_text = responder_com_contexto(question)
            metrics = coletar_metricas_rag(
                question=question, answer=answer,
                retrieved_chunks=fused, context_text=context_text,
                ground_truth_answer=gt_a, ground_truth_contexts=gt_c,
            )
            result = {
                "id": qid,
                "timestamp": datetime.utcnow().isoformat(),
                "question": question,
                "answer": answer,
                "meta": meta,
                "ground_truth_answer":   gt_a,
                "ground_truth_contexts": gt_c,
                "retrieved_chunks": [
                    {
                        "chunk_id":       c.get("chunk_id", ""),
                        "source":         c.get("source", ""),
                        "content":        c.get("content", ""),
                        "score_cosine":   c.get("score_cosine"),
                        "rrf_score":      c.get("rrf_score"),
                        "reranker_score": c.get("reranker_score"),
                    }
                    for c in fused
                ],
                "metrics": metrics,
            }
            fo.write(json.dumps(result, ensure_ascii=False) + "\n")
    print(f"✓ Dataset avaliado. Resultados em: {out_path}")

print("OK: avaliar_dataset pronto")

## 🔍 Patch — Filtro Estruturado por Tipo/Ano

In [ ]:
# ── Filtro Estruturado no HyPA-RAG (patch v13) ─────────────
# Só dispara filtro quando o tipo é citado de forma ESPECÍFICA
# "lei do processo" NÃO filtra — "lei nº 14133" SIM
TIPO_KEYWORDS = {
    "instrucao_normativa": [
        "instrução normativa", "instrucao normativa", "normativa"
    ],
    "lei": [
        "lei nº", "lei no ", "lei n.", "lei federal",
        "lei complementar", "lei ordinaria", "lei estadual"
    ],
    "decreto": [
        "decreto nº", "decreto no ", "decreto n.",
        "decreto-lei", "decreto lei"
    ],
    "documento": [
        "portaria", "resolucao", "circular", "nota tecnica"
    ],
}

def detect_tipo(query: str) -> str | None:
    q = query.lower()
    for tipo, keywords in TIPO_KEYWORDS.items():
        if any(kw in q for kw in keywords):
            return tipo
    return None

def detect_ano_range(query: str) -> tuple | None:
    match = re.search(r"entre\s+(20\d{2})\s*(e|a|até|ate)\s*(20\d{2})", query)
    if match:
        return int(match.group(1)), int(match.group(3))
    match = re.search(r"(20\d{2})\s*(a|até|ate|-)\s*(20\d{2})", query)
    if match:
        return int(match.group(1)), int(match.group(3))
    match = re.search(r"(desde|após|apos|a partir de)\s+(20\d{2})", query)
    if match:
        return int(match.group(2)), 2026
    match = re.search(r"\b(20\d{2})\b", query)
    if match:
        year = int(match.group(1))
        return year, year
    return None

def detectar_source_na_pergunta(pergunta: str) -> str | None:
    p = pergunta.strip()

    # aceita "fonte: L12527_lai.pdf" ou "source=L12527_lai.pdf"
    m = re.search(r'\b(?:fonte|source)\s*[:=]\s*([^\s]+)', p, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # se a pessoa digitar algo que parece arquivo pdf na pergunta
    m2 = re.search(r'([A-Za-z0-9_\-]+\.pdf)\b', p, flags=re.IGNORECASE)
    if m2:
        return m2.group(1).strip()

    return None


def build_filter(query: str, area: str | None = None, source: str | None = None) -> str:
    parts = []

    # SOURCE (documento específico)
    src_eff = source or detectar_source_na_pergunta(query)
    if src_eff:
        parts.append(f"@source:{{{escape_tag_value(src_eff)}}}")

    # ÁREA jurídica
    area_detected, conf, _ = detectar_area_na_pergunta(query)
    area_eff = area or (area_detected if conf >= 0.55 else None)  # ← fix: desempacota tupla
    if area_eff:
        parts.append(f"@area:{{{area_eff}}}")

    # Tipo normativo
    tipo = detect_tipo(query)
    if tipo:
        parts.append(f"@tipo:{{{tipo}}}")

    # Ano
    ano_range = detect_ano_range(query)
    if ano_range:
        parts.append(f"@ano:[{ano_range[0]} {ano_range[1]}]")

    return " ".join(parts)


def knn_dense_filtrado(pergunta: str, k: int, score_threshold: float = 0.50, filtro: str = ""):
    emb = encode_queries([pergunta])[0]
    vec = np.array(emb, dtype=EMBED_DTYPE).tobytes()

    base = "*"
    if filtro:
        base = f"({filtro})"   # <-- aqui é a correção

    q = (
        Query(f"{base}=>[KNN {k} @embedding $vec AS score]")
        .sort_by("score")
        .paging(0, k)
        .return_fields("content","source","chunk_id","tipo","ano","numero","artigo","area","score")
        .dialect(2)
    )

    res = r.ft(REDIS_INDEX).search(q, query_params={"vec": vec})
    print(f"[DEBUG knn_dense_filtrado] q='{q.query_string()}' docs={len(getattr(res,'docs',[]))}")

    out = []
    for d in getattr(res, "docs", []):
        raw_score  = float(getattr(d, "score", 1.0))  # distância COSINE (0..2)
        cosine_sim = 1.0 - (raw_score / 2.0)

        if cosine_sim < score_threshold:
            continue

        out.append({
            "redis_id":     d.id,
            "content":      as_text(getattr(d, "content",  "")),
            "source":       as_text(getattr(d, "source",   "")),
            "chunk_id":     as_text(getattr(d, "chunk_id", "")),
            "tipo":         as_text(getattr(d, "tipo",     "")),
            "ano":          as_text(getattr(d, "ano",      "")),
            "numero":       as_text(getattr(d, "numero",   "")),
            "artigo":       as_text(getattr(d, "artigo",   "")),
            "area":         as_text(getattr(d, "area",     "")),
            "score_cosine": cosine_sim,
        })

    return out

def sparse_text_filtrado(pergunta: str, k: int, filtro: str = ""):
    # 1) tenta usar termos-chave, mas sem matar recall
    termos = extrair_termos_chave(pergunta) or ""
    if len(termos.split()) < 2:
        termos = pergunta

    def _run(texto_base: str):
        texto = escape_redisearch_query(texto_base)
        texto = texto if texto else "*"

        # sempre parentetiza pra evitar ambiguidades/sintaxe
        if filtro:
            query_str = f"({filtro}) ({texto})"
        else:
            query_str = f"({texto})" if texto != "*" else "*"

        q = (
            Query(query_str)
            .paging(0, k)
            .return_fields("content","source","chunk_id","tipo","ano","numero","artigo","area")
            .dialect(2)
        )
        res = r.ft(REDIS_INDEX).search(q)
        print(f"[DEBUG sparse_text_filtrado] q='{query_str}' docs={len(getattr(res,'docs',[]))}")

        out = []
        for d in getattr(res, "docs", []):
            out.append({
                "redis_id":     d.id,
                "content":      as_text(getattr(d, "content",  "")),
                "source":       as_text(getattr(d, "source",   "")),
                "chunk_id":     as_text(getattr(d, "chunk_id", "")),
                "tipo":         as_text(getattr(d, "tipo",     "")),
                "ano":          as_text(getattr(d, "ano",      "")),
                "numero":       as_text(getattr(d, "numero",   "")),
                "artigo":       as_text(getattr(d, "artigo",   "")),
                "area":         as_text(getattr(d, "area",     "")),
                "score_cosine": None,  # ← fix: sparse não tem score vetorial
            })
        return out

    # tentativa 1: termos-chave
    try:
        return _run(termos)
    except Exception as e1:
        print(f"[WARN] sparse_filtrado termos_chave falhou: {e1}")

    # tentativa 2: pergunta inteira (mantém filtro!)
    try:
        return _run(pergunta)
    except Exception as e2:
        print(f"[WARN] sparse_filtrado pergunta inteira falhou: {e2}")

    # tentativa 3: sem filtro (último recurso)
    try:
        return _run(pergunta if pergunta else "*")
    except Exception as e3:
        print(f"[WARN] sparse_filtrado fallback sem filtro falhou: {e3}")
        return []

def buscar_contexto_hypa_filtrado(
    pergunta: str,
    area: str | None = None,
    source: str | None = None,
    score_threshold: float | None = None,
):
    """
    Retrieval HyPA-RAG com filtros estruturais do v13.

    Permite:
    - filtrar por área jurídica (TAG @area)
    - filtrar por documento específico (TAG @source)
    - ainda usar detecção automática se não passar nada

    Ex:
        buscar_contexto_hypa_filtrado(
            "o que diz o art. 10?",
            source="L12527_lai.pdf"
        )
    """

    # --------------------------------------------------
    # 1️⃣ Classificação de complexidade (pipeline original)
    # --------------------------------------------------
    nivel = classificar_complexidade(pergunta)
    p     = params_por_complexidade(nivel)

    # --------------------------------------------------
    # 2️⃣ Threshold adaptativo
    # --------------------------------------------------
    if score_threshold is None:
        score_threshold = _SCORE_THRESHOLD_BY_NIVEL.get(nivel, 0.60)

    # --------------------------------------------------
    # 3️⃣ Monta filtro estruturado do Redis (V13)
    # --------------------------------------------------
    filtro = build_filter(
        pergunta,
        area=area,
        source=source,
    )

    print(f"[HyPA-RAG] nivel={nivel} | filtro={filtro or '(nenhum)'}")

    # --------------------------------------------------
    # 4️⃣ Dense Retrieval (vetorial)
    # --------------------------------------------------
    dense = knn_dense_filtrado(
        pergunta,
        k=p["k_dense"],
        score_threshold=score_threshold,
        filtro=filtro,
    )

    # --------------------------------------------------
    # 5️⃣ Sparse Retrieval (BM25)
    # --------------------------------------------------
    sparse = sparse_text_filtrado(
        pergunta,
        k=p["k_sparse"],
        filtro=filtro,
    )

    # --------------------------------------------------
    # 6️⃣ Fusão híbrida (RRF)
    # --------------------------------------------------
    fused_pool = rrf_fusion([dense, sparse], pesos=[2.0, 1.0])

    # --------------------------------------------------
    # 7️⃣ Seleção final de contexto
    # --------------------------------------------------
    max_doc = min(2, max(1, p["k_final"] - 1))

    fused = selecionar_contexto_final(
        fused_pool,
        k_final=p["k_final"],
        pergunta=pergunta,
        max_chars=9000,
        min_rrf_score=0.003,
        max_chunks_por_doc=max_doc,
    )

    # --------------------------------------------------
    # 8️⃣ Monta contexto textual
    # --------------------------------------------------
    trechos = [
        f"[Fonte: {it['source']} | Chunk: {it['chunk_id']}]\n{it['content']}"
        for it in fused
    ]

    contexto = "\n\n---\n\n".join(trechos)

    meta = {
        "nivel": nivel,
        "filtro": filtro or "(nenhum)",
        "chunks": len(fused),           # ← fix: qtd de chunks efetivamente usados
        "area_aplicada": area,
        "source_aplicado": source,
        **p,
    }

    return contexto, meta, fused
print("OK: buscar_contexto_hypa_filtrado pronto")


## 🌐 Flask — Rotas e Front-end

In [ ]:
# ── 9. Flask — Front-end ─────────────────────────────────────
app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>HyPA-RAG</title>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      font-family: 'Segoe UI', sans-serif;
      background: #f0f4f8;
      min-height: 100vh;
      display: flex;
      flex-direction: column;
      align-items: center;
      padding: 2rem 1rem;
    }
    header {
      width: 100%;
      max-width: 800px;
      margin-bottom: 1.5rem;
      text-align: center;
    }
    header h1 { color: #1a365d; font-size: 1.8rem; }
    header p  { color: #4a5568; font-size: 0.95rem; margin-top: 4px; }

    .card {
      background: #fff;
      border-radius: 12px;
      box-shadow: 0 2px 12px rgba(0,0,0,.1);
      padding: 1.5rem;
      width: 100%;
      max-width: 800px;
      margin-bottom: 1rem;
    }

    textarea {
      width: 100%;
      min-height: 80px;
      border: 1px solid #cbd5e0;
      border-radius: 8px;
      padding: 0.75rem;
      font-size: 1rem;
      resize: vertical;
      outline: none;
    }
    textarea:focus { border-color: #4299e1; box-shadow: 0 0 0 3px rgba(66,153,225,.2); }

    .row { display: flex; gap: 0.75rem; margin-top: 0.75rem; align-items: center; flex-wrap: wrap; }
    .filter-row { display: flex; gap: 0.5rem; margin-top: 0.5rem; flex-wrap: wrap; }
    .filter-row select, .filter-row input {
      border: 1px solid #cbd5e0; border-radius: 6px;
      padding: 0.35rem 0.6rem; font-size: 0.85rem;
      color: #2d3748; background: #f7fafc; outline: none;
    }
    .filter-row select:focus, .filter-row input:focus {
      border-color: #4299e1; box-shadow: 0 0 0 2px rgba(66,153,225,.2);
    }
    .filter-label { font-size: 0.78rem; color: #718096; align-self: center; }

    button {
      background: #2b6cb0;
      color: #fff;
      border: none;
      border-radius: 8px;
      padding: 0.6rem 1.4rem;
      font-size: 1rem;
      cursor: pointer;
      transition: background .2s;
    }
    button:hover   { background: #2c5282; }
    button:disabled{ background: #90cdf4; cursor: not-allowed; }

    .badge {
      font-size: 0.78rem;
      padding: 3px 10px;
      border-radius: 20px;
      font-weight: 600;
    }
    .badge-simples  { background: #c6f6d5; color: #22543d; }
    .badge-media    { background: #fefcbf; color: #744210; }
    .badge-complexa { background: #fed7d7; color: #742a2a; }

    .answer-box {
      background: #edf2f7;
      border-left: 4px solid #4299e1;
      border-radius: 0 8px 8px 0;
      padding: 1rem;
      white-space: pre-wrap;
      font-size: 0.95rem;
      line-height: 1.6;
      min-height: 60px;
    }

    .metrics-grid {
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(160px, 1fr));
      gap: 0.6rem;
      margin-top: 0.75rem;
    }
    .metric-item {
      background: #ebf8ff;
      border-radius: 8px;
      padding: 0.5rem 0.75rem;
      text-align: center;
    }
    .metric-item .label { font-size: 0.72rem; color: #2b6cb0; font-weight: 600; text-transform: uppercase; }
    .metric-item .value { font-size: 1.2rem; font-weight: 700; color: #1a365d; }

    .sources { margin-top: 0.75rem; }
    .source-chip {
      display: inline-block;
      background: #e2e8f0;
      border-radius: 4px;
      padding: 2px 8px;
      font-size: 0.8rem;
      margin: 2px;
      color: #2d3748;
    }

    details summary { cursor: pointer; color: #2b6cb0; font-size: 0.9rem; margin-top: 0.5rem; }
    details pre {
      background: #1a202c;
      color: #e2e8f0;
      border-radius: 8px;
      padding: 1rem;
      font-size: 0.8rem;
      overflow-x: auto;
      margin-top: 0.5rem;
      max-height: 300px;
    }

    .spinner { display: none; margin-left: 0.5rem; }
    .spinner.active { display: inline-block; }
    @keyframes spin { to { transform: rotate(360deg); } }
    .spinner svg { animation: spin 1s linear infinite; vertical-align: middle; }

    .error { color: #c53030; background: #fff5f5; border-left: 4px solid #fc8181;
             padding: 0.75rem; border-radius: 0 8px 8px 0; font-size: 0.9rem; }
  </style>
</head>
<body>

<header>
  <h1>🔍 HyPA-RAG</h1>
  <p>Consulta inteligente a documentos normativos · Retrieval Híbrido + Avaliação de Métricas</p>
</header>

<div class="card">
  <textarea id="pergunta" placeholder="Digite sua pergunta sobre os documentos…"></textarea>
  <div class="filter-row">
    <span class="filter-label">Área:</span>
    <select id="filtro-area">
      <option value="">Detectar automaticamente</option>
      <option value="transparencia_controle">Transparência e Controle</option>
      <option value="processo_administrativo">Processo Administrativo</option>
      <option value="servidor_publico">Servidor Público</option>
      <option value="licitacoes_contratos">Licitações e Contratos</option>
      <option value="financas_publicas">Finanças Públicas</option>
      <option value="organizacao_administrativa">Organização Administrativa</option>
      <option value="educacao">Educação</option>
      <option value="instrumentos_planejamento">Planejamento</option>
      <option value="norma_geral">Norma Geral</option>
      <option value="outros">Outros</option>
    </select>
    <span class="filter-label">Documento:</span>
    <input id="filtro-source" type="text" placeholder="ex: L12527_lai.pdf" style="width:180px">
  </div>
  <div class="row">
    <button id="btnPerguntar" onclick="perguntar()">
      Perguntar
      <span class="spinner" id="spinner">
        <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor"
             stroke-width="3" stroke-linecap="round">
          <path d="M12 2v4M12 18v4M4.93 4.93l2.83 2.83M16.24 16.24l2.83 2.83
                   M2 12h4M18 12h4M4.93 19.07l2.83-2.83M16.24 7.76l2.83-2.83"/>
        </svg>
      </span>
    </button>
    <span id="nivel-badge" class="badge" style="display:none"></span>
  </div>
</div>

<div class="card" id="resultado" style="display:none">
  <h3 style="margin-bottom:.75rem;color:#2d3748">Resposta</h3>
  <div class="answer-box" id="resposta-texto"></div>

  <div class="sources" id="fontes-area"></div>

  <div id="metricas-area" style="margin-top:1rem;display:none">
    <h4 style="color:#2d3748;margin-bottom:.4rem">Métricas RAG</h4>
    <div class="metrics-grid" id="metricas-grid"></div>
  </div>

  <details id="detalhes">
    <summary>Ver JSON completo</summary>
    <pre id="json-raw"></pre>
  </details>
</div>

<div class="card error" id="erro-box" style="display:none"></div>

<script>
const METRICAS_LABELS = {
  hit_rate:          "Hit Rate",
  mrr:               "MRR",
  ndcg:              "nDCG",
  context_precision: "Ctx Precision",
  context_recall:    "Ctx Recall",
  faithfulness:      "Faithfulness",
  answer_relevance:  "Ans Relevance",
  answer_correctness:"Ans Correctness",
};

async function perguntar() {
  const pergunta = document.getElementById("pergunta").value.trim();
  if (!pergunta) return;

  // UI loading
  document.getElementById("btnPerguntar").disabled = true;
  document.getElementById("spinner").classList.add("active");
  document.getElementById("resultado").style.display = "none";
  document.getElementById("erro-box").style.display  = "none";
  document.getElementById("nivel-badge").style.display = "none";

  try {
    const area   = document.getElementById("filtro-area").value.trim();
    const source = document.getElementById("filtro-source").value.trim();
    const resp = await fetch("/perguntar", {
      method: "POST",
      headers: {"Content-Type": "application/json"},
      body: JSON.stringify({pergunta, area: area || null, source: source || null})
    });

    const data = await resp.json();
    if (!resp.ok) throw new Error(data.erro || "Erro desconhecido");

    // Resposta
    document.getElementById("resposta-texto").textContent = data.resposta;

    // Badge de complexidade
    const badge = document.getElementById("nivel-badge");
    const nivel = data.meta?.nivel || "";
    const filtro = data.meta?.filtro && data.meta.filtro !== "(nenhum)" ? " · " + data.meta.filtro : "";
    badge.textContent = "Complexidade: " + nivel + filtro;
    badge.className   = "badge badge-" + nivel;
    badge.style.display = "inline-block";

    // Fontes
    const fontes = [...new Set((data.fontes || []).map(f => f.source).filter(Boolean))];
    const fontesArea = document.getElementById("fontes-area");
    if (fontes.length) {
      fontesArea.innerHTML = "<strong style='font-size:.85rem;color:#4a5568'>Fontes:</strong> " +
        fontes.map(s => `<span class='source-chip'>${s}</span>`).join("");
    } else {
      fontesArea.innerHTML = "";
    }

    // Métricas
    const m = data.metricas;
    if (m) {
      const grid = document.getElementById("metricas-grid");
      grid.innerHTML = "";
      Object.entries(METRICAS_LABELS).forEach(([key, label]) => {
        const val = m[key];
        if (val === null || val === undefined) return;
        const item = document.createElement("div");
        item.className = "metric-item";
        item.innerHTML = `<div class="label">${label}</div>
                          <div class="value">${(typeof val === "number" ? val.toFixed(3) : val)}</div>`;
        grid.appendChild(item);
      });
      document.getElementById("metricas-area").style.display = "block";
    } else {
      document.getElementById("metricas-area").style.display = "none";
    }

    // JSON raw
    document.getElementById("json-raw").textContent = JSON.stringify(data, null, 2);
    document.getElementById("resultado").style.display = "block";

  } catch(e) {
    document.getElementById("erro-box").textContent = "Erro: " + e.message;
    document.getElementById("erro-box").style.display = "block";
  } finally {
    document.getElementById("btnPerguntar").disabled = false;
    document.getElementById("spinner").classList.remove("active");
  }
}

// Enviar com Ctrl+Enter
document.getElementById("pergunta").addEventListener("keydown", e => {
  if (e.ctrlKey && e.key === "Enter") perguntar();
});
</script>
</body>
</html>
"""

@app.route("/")
def index():
    return render_template_string(HTML_TEMPLATE)


@app.route("/perguntar", methods=["POST"])
def rota_perguntar():
    data = request.get_json(force=True)
    pergunta = (data.get("pergunta") or "").strip()
    area     = (data.get("area")     or "").strip() or None   # ← fix: filtro explícito
    source   = (data.get("source")   or "").strip() or None   # ← fix: filtro explícito
    if not pergunta:
        return jsonify({"erro": "Pergunta vazia"}), 400

    try:
        answer, meta, fused, context_text = responder_com_contexto(pergunta, area=area, source=source)

        metricas = None
        if RAG_EVAL_ENABLE:
            try:
                metricas = coletar_metricas_rag(
                    question=pergunta,
                    answer=answer,
                    retrieved_chunks=fused,
                    context_text=context_text,
                )
                metricas_ui = {k: v for k, v in metricas.items()
                               if k != "context_precision_details"}
                log_metricas({
                    "timestamp": datetime.utcnow().isoformat(),
                    "question": pergunta,
                    "answer": answer,
                    "meta": meta,
                    "retrieved_chunks": [
                        {
                            "chunk_id":       c.get("chunk_id", ""),
                            "source":         c.get("source",   ""),
                            "content":        c.get("content",  ""),
                            "tipo":           c.get("tipo",     ""),   # ← v13
                            "ano":            c.get("ano",      ""),   # ← v13
                            "area":           c.get("area",     ""),   # ← v13
                            "score_cosine":   c.get("score_cosine"),
                            "rrf_score":      c.get("rrf_score"),
                            "reranker_score": c.get("reranker_score"),
                        }
                        for c in fused
                    ],
                    "metrics": metricas,
                })
            except Exception as e:
                print(f"[WARN] Métricas falharam: {e}")
                metricas_ui = None
        else:
            metricas_ui = None

        return jsonify({
            "resposta": answer,
            "meta":     meta,
            "fontes": [
                {
                    "chunk_id":       c.get("chunk_id", ""),
                    "source":         c.get("source",   ""),
                    "content":        c.get("content",  ""),   # ← fix: texto do chunk
                    "tipo":           c.get("tipo",     ""),
                    "ano":            c.get("ano",      ""),
                    "area":           c.get("area",     ""),
                    "score_cosine":   c.get("score_cosine"),
                    "rrf_score":      c.get("rrf_score"),
                    "reranker_score": c.get("reranker_score"),
                }
                for c in fused
            ],
            "metricas": metricas_ui,
        })

    except Exception as e:
        return jsonify({"erro": str(e)}), 500



@app.route("/health")
def health():
    return jsonify({"status": "ok", "model": MODEL_NAME})


# ── Inicialização ────────────────────────────────────────────

## Configuração da Reindexação

In [ ]:
# =========================

# CONFIG
# =========================

@dataclass
class Config:
    # Redis
    redis_host: str = os.getenv("REDIS_HOST", "localhost")
    redis_port: int = int(os.getenv("REDIS_PORT", "6379"))
    redis_password: Optional[str] = os.getenv("REDIS_PASSWORD", None)
    redis_db: int = int(os.getenv("REDIS_DB", "0"))

    # Índices — mesmas variáveis de env do app para evitar dessincronização
    index_v13:  str = REDIS_INDEX   # ← fix: era os.getenv("REDIS_INDEX_V13")
    prefix_v13: str = REDIS_PREFIX  # ← fix: era os.getenv("REDIS_PREFIX_V13")

    # Pasta PDFs
    pdf_dir: str = os.getenv("PASTA_PDFS", "./documentos")

    # Embeddings
    embedding_model_name: str = os.getenv("EMBEDDING_MODEL_NAME", "intfloat/e5-base-v2")
    embed_dim: int = int(os.getenv("EMBED_DIM", "768"))   # e5-base-v2 = 768
    batch_size: int = int(os.getenv("EMBED_BATCH_SIZE", "32"))

    # Triagem anti-scan
    min_chars_pdf: int = int(os.getenv("MIN_CHARS_PDF", "300"))

    # Chunking (artigo)
    min_chars_chunk: int = int(os.getenv("MIN_CHARS_CHUNK", "250"))
    max_chars_chunk: int = int(os.getenv("MAX_CHARS_CHUNK", "3500"))
    fallback_split_regex: str = r"(?=(T[ÍI]TULO|CAP[ÍI]TULO|SEÇÃO|SUBSEÇÃO)\s+[IVXLC0-9]+)"

    # Filtro de "tabela quebrada"
    table_alpha_ratio_threshold: float = float(os.getenv("TABLE_ALPHA_RATIO", "0.55"))
    max_single_char_lines_ratio: float = float(os.getenv("SINGLE_CHAR_LINES_RATIO", "0.25"))

    # Logs / manifesto
    out_dir: str = os.getenv("OUT_DIR", "./reindex_v13_out")


CFG = Config()

## Helpers — Reindexação

In [7]:
# =========================
# HELPERS — MANIFESTO
# =========================

def file_fingerprint(p: Path) -> dict:
    st = p.stat()
    return {"size": st.st_size, "mtime": int(st.st_mtime)}

def load_manifest(path: Path) -> dict:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}

def save_manifest(path: Path, data: dict) -> None:
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


# =========================
# HELPERS — UTILIDADES
# =========================

def safe_mkdir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)

def sha1_text(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def connect_redis(cfg: Config) -> redis.Redis:
    r = redis.Redis(
        host=cfg.redis_host,
        port=cfg.redis_port,
        password=cfg.redis_password,
        db=cfg.redis_db,
        decode_responses=False,
    )
    r.ping()
    return r


# =========================
# HELPERS — PROCESSAMENTO
# =========================

def extract_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    out: List[str] = []
    for page in reader.pages:
        try:
            out.append(page.extract_text() or "")
        except Exception:
            out.append("")
    return "\n".join(out).strip()

def normalize_text(text: str) -> str:
    text = re.sub(r"-\s*\n\s*", "", text)           # desfaz hifenização em quebra de linha
    text = re.sub(r"\r\n?", "\n", text)             # normaliza quebras
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)          # tabs e espaços múltiplos
    text = re.sub(r"[ \t]+\n", "\n", text)          # espaços ao redor de quebras
    text = re.sub(r"\n[ \t]+", "\n", text)
    return text.strip()

def alpha_ratio(s: str) -> float:
    if not s:
        return 0.0
    letters = sum(ch.isalpha() for ch in s)
    return letters / max(1, len(s))

def single_char_lines_ratio(s: str) -> float:
    lines = [ln.strip() for ln in s.splitlines() if ln.strip()]
    if not lines:
        return 0.0
    single = sum(1 for ln in lines if len(ln) == 1)
    return single / len(lines)

def is_table_like(chunk: str, cfg: Config) -> bool:
    ar = alpha_ratio(chunk)
    scr = single_char_lines_ratio(chunk)
    return (ar < cfg.table_alpha_ratio_threshold) or (scr > cfg.max_single_char_lines_ratio)

def extract_basic_metadata(text: str, filename: str) -> Dict[str, str]:
    meta: Dict[str, str] = {}
    head = text[:1500]

    if re.search(r"\bLEI\b", head, flags=re.IGNORECASE):
        meta["tipo"] = "lei"
    elif re.search(r"INSTRUÇÃO\s+NORMATIVA", head, flags=re.IGNORECASE):
        meta["tipo"] = "instrucao_normativa"
    elif re.search(r"\bDECRETO\b", head, flags=re.IGNORECASE):
        meta["tipo"] = "decreto"
    else:
        meta["tipo"] = "documento"

    m = re.search(r"(LEI|DECRETO|INSTRUÇÃO\s+NORMATIVA)[^\n]{0,90}?\bN[º°]?\s*([0-9\.\-_/]+)", head, flags=re.IGNORECASE)
    if m:
        meta["numero"] = m.group(2).strip()

    m2 = re.search(r"\bde\s+\d{1,2}\s+de\s+[A-Za-zçÇáéíóúãõÁÉÍÓÚÃÕ]+\s+de\s+(\d{4})\b", head, flags=re.IGNORECASE)
    if m2:
        meta["ano"] = m2.group(1)
    else:
        m3 = re.search(r"\b(19|20)\d{2}\b", head)
        if m3:
            meta["ano"] = m3.group(0)

    meta["arquivo"] = filename
    return meta

def split_by_articles(text: str, cfg: Config) -> List[Tuple[str, str]]:
    pattern = r"(?=(Art\.\s*\d+\s*º?))"
    parts = re.split(pattern, text)

    if len(parts) > 1:
        chunks: List[Tuple[str, str]] = []
        prefix = parts[0].strip()
        if prefix and len(prefix) >= cfg.min_chars_chunk:
            chunks.append(("PREAMBULO", prefix))

        i = 1
        while i < len(parts) - 1:
            art_id = parts[i].strip()
            body = parts[i + 1].strip()
            art_text = f"{art_id}\n{body}".strip()
            if art_text:
                chunks.append((art_id, art_text))
            i += 2
        return chunks

    # fallback por capítulos/seções
    fparts = re.split(cfg.fallback_split_regex, text, flags=re.IGNORECASE)
    out: List[Tuple[str, str]] = []
    buf0 = fparts[0].strip()
    if buf0 and len(buf0) >= cfg.min_chars_chunk:
        out.append(("BLOCO_0", buf0))

    idx = 1
    block_n = 1
    while idx < len(fparts) - 1:
        marker = (fparts[idx] or "").strip()
        body = (fparts[idx + 1] or "").strip()
        combined = (marker + "\n" + body).strip()
        if combined and len(combined) >= cfg.min_chars_chunk:
            out.append((f"BLOCO_{block_n}", combined))
            block_n += 1
        idx += 2

    if not out and text.strip():
        out = [("TEXTO_INTEGRAL", text.strip())]

    return out

def post_process_chunks(chunks: List[Tuple[str, str]], cfg: Config) -> List[Tuple[str, str]]:
    cleaned: List[Tuple[str, str]] = []
    for cid, ctext in chunks:
        ctext = ctext.strip()
        if len(ctext) < cfg.min_chars_chunk:
            continue

        # fatiar chunks gigantes por parágrafos (mantém semântica)
        if len(ctext) > cfg.max_chars_chunk:
            paras = [p.strip() for p in re.split(r"\n{2,}", ctext) if p.strip()]
            buff = ""
            part = 1
            for p in paras:
                if len(buff) + len(p) + 2 <= cfg.max_chars_chunk:
                    buff = (buff + "\n\n" + p).strip() if buff else p
                else:
                    if len(buff) >= cfg.min_chars_chunk:
                        cleaned.append((f"{cid}_p{part}", buff))
                        part += 1
                    buff = p
            if buff and len(buff) >= cfg.min_chars_chunk:
                cleaned.append((f"{cid}_p{part}", buff))
        else:
            cleaned.append((cid, ctext))
    return cleaned


# =========================
# HELPERS — EMBEDDINGS / REDIS
# =========================

def embed_passages(model: SentenceTransformer, texts: List[str], cfg: Config) -> np.ndarray:
    prefixed = [f"passage: {t}" for t in texts]
    emb = model.encode(
        prefixed,
        batch_size=cfg.batch_size,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return np.asarray(emb, dtype=np.float32)

def ensure_index_v13_hash(r: redis.Redis, cfg: Config) -> None:
    try:
        r.ft(cfg.index_v13).info()
        print(f"[OK] Índice já existe: {cfg.index_v13}")
        return
    except Exception:
        pass

    schema = (
        # Campos que o seu app usa:
        TextField("content"),
        TagField("source"),
        TextField("chunk_id"),
        VectorField(
            "embedding",
            "HNSW",
            {
                "TYPE": "FLOAT32",
                "DIM": cfg.embed_dim,
                "DISTANCE_METRIC": "COSINE",
                "M": 16,
                "EF_CONSTRUCTION": 200,
            },
        ),
        # Extras (não quebram o app):
        TagField("tipo"),
        TagField("numero"),
        NumericField("ano"),
        TagField("artigo"),
        TagField("area"),  # ← fix: campo obrigatório para filtro @area
    )

    definition = IndexDefinition(prefix=[cfg.prefix_v13], index_type=IndexType.HASH)
    r.ft(cfg.index_v13).create_index(schema, definition=definition)
    print(f"[OK] Índice criado: {cfg.index_v13}  (prefix='{cfg.prefix_v13}')")

def index_records_hash(
    r: redis.Redis,
    cfg: Config,
    records: List[Dict],
    embeddings: np.ndarray,
) -> int:
    pipe = r.pipeline(transaction=False)

    for rec, vec in zip(records, embeddings):
        key = f"{cfg.prefix_v13}{rec['doc_id']}".encode("utf-8")
        mapping = {
            b"content":   rec["content"].encode("utf-8", errors="ignore"),
            b"source":    rec["source"].encode("utf-8", errors="ignore"),
            b"chunk_id":  rec["chunk_id"].encode("utf-8", errors="ignore"),
            b"embedding": vec.astype(np.float32).tobytes(),
            b"tipo":      rec.get("tipo", "").encode("utf-8", errors="ignore"),
            b"numero":    rec.get("numero", "").encode("utf-8", errors="ignore"),
            b"ano":       str(rec.get("ano", "0") or "0").encode("utf-8", errors="ignore"),
            b"artigo":    rec.get("artigo", "").encode("utf-8", errors="ignore"),
            b"area":      rec.get("area", "").encode("utf-8", errors="ignore"),
        }
        pipe.hset(key, mapping=mapping)

    pipe.execute()
    return len(records)


# =========================
# HELPERS — PIPELINE PDF
# =========================

def build_records_from_pdf(pdf_path: Path, cfg: Config) -> Tuple[List[Dict], Dict]:
    raw = extract_pdf_text(pdf_path)
    stats = {
        "file": pdf_path.name,
        "raw_chars": len(raw),
        "status": "OK",
        "reason": "",
        "chunks_total": 0,
        "chunks_indexed": 0,
        "chunks_dropped_table": 0,
        "chunks_dropped_short": 0,
    }

    if len(raw.strip()) < cfg.min_chars_pdf:
        stats["status"] = "SCANNED_OR_EMPTY"
        stats["reason"] = f"texto_extraido<{cfg.min_chars_pdf}"
        return [], stats

    text = normalize_text(raw)
    meta = extract_basic_metadata(text, pdf_path.name)

    chunks = split_by_articles(text, cfg)
    chunks = post_process_chunks(chunks, cfg)
    stats["chunks_total"] = len(chunks)

    records: List[Dict] = []
    for cid, chunk_text in chunks:
        if len(chunk_text) < cfg.min_chars_chunk:
            stats["chunks_dropped_short"] += 1
            continue
        if is_table_like(chunk_text, cfg):
            stats["chunks_dropped_table"] += 1
            continue

        # doc_id estável baseado no conteúdo — não muda entre execuções
        doc_id = sha1_text(f"{pdf_path.name}||{cid}||{chunk_text}")

        records.append({
            "doc_id":  doc_id,
            "source":  pdf_path.name,
            "chunk_id": f"{pdf_path.name}__{cid}",
            "content": chunk_text,
            "tipo":    meta.get("tipo", "documento"),
            "numero":  meta.get("numero", ""),
            "ano":     meta.get("ano", "0"),
            "artigo":  cid,
            "area":    inferir_area(chunk_text, pdf_path.name),
        })

    stats["chunks_indexed"] = len(records)
    if not records and stats["status"] == "OK":
        stats["status"] = "NO_VALID_CHUNKS"
        stats["reason"] = "todos_chunks_filtrados(table/short)"

    return records, stats

## 🚀 Main — Reindexar PDFs novos e iniciar servidor

In [ ]:
# =========================
# MAIN — Reindexação + Flask
# =========================

def main() -> None:
    safe_mkdir(CFG.out_dir)

    audit_path    = Path(CFG.out_dir) / "v13_clean_audit_report.jsonl"
    summary_path  = Path(CFG.out_dir) / "v13_clean_summary.json"
    manifest_path = Path(CFG.out_dir) / "v13_clean_manifest.json"

    r_redis = connect_redis(CFG)

    print(f"[INFO] Índice V13: {CFG.index_v13}  |  Prefixo: {CFG.prefix_v13}")
    print(f"[INFO] Pasta PDFs: {Path(CFG.pdf_dir).resolve()}")

    ensure_index_v13_hash(r_redis, CFG)

    pdf_dir   = Path(CFG.pdf_dir)
    pdf_files = sorted([p for p in pdf_dir.glob("**/*.pdf") if p.is_file()])
    if not pdf_files:
        print(f"[WARN] Nenhum PDF encontrado em: {pdf_dir.resolve()}")
    else:
        previous_manifest = load_manifest(manifest_path)
        current_manifest  = {p.name: file_fingerprint(p) for p in pdf_files}
        pending = [p for p in pdf_files
                   if current_manifest[p.name] != previous_manifest.get(p.name)]

        if not pending:
            print("\n[OK] Nenhum PDF novo ou alterado — reindexação não necessária.\n")
        else:
            print(f"\n[INFO] {len(pending)} PDF(s) para indexar (de {len(pdf_files)} total).\n")
            print(f"[MODEL] Carregando embeddings: {CFG.embedding_model_name}")
            model = SentenceTransformer(CFG.embedding_model_name)

            t0 = time.time()
            totals = {"total_files": 0, "total_scanned_or_empty": 0,
                      "total_no_valid_chunks": 0, "total_chunks_seen": 0,
                      "total_chunks_indexed": 0, "seconds": 0.0}

            with open(audit_path, "a", encoding="utf-8") as fout:
                for pdf in tqdm(pending, desc="Reindex v13_clean (HASH)"):
                    totals["total_files"] += 1
                    records, stats = build_records_from_pdf(pdf, CFG)
                    fout.write(json.dumps(stats, ensure_ascii=False) + "\n")

                    if stats["status"] in ("SCANNED_OR_EMPTY", "NO_VALID_CHUNKS"):
                        totals["total_" + ("scanned_or_empty" if stats["status"] == "SCANNED_OR_EMPTY" else "no_valid_chunks")] += 1
                        previous_manifest[pdf.name] = current_manifest[pdf.name]
                        continue
                    if not records:
                        continue

                    embs    = embed_passages(model, [rec["content"] for rec in records], CFG)
                    indexed = index_records_hash(r_redis, CFG, records, embs)
                    totals["total_chunks_seen"]    += stats["chunks_total"]
                    totals["total_chunks_indexed"] += indexed
                    previous_manifest[pdf.name] = current_manifest[pdf.name]
                    save_manifest(manifest_path, previous_manifest)

            totals["seconds"] = round(time.time() - t0, 2)
            save_manifest(manifest_path, previous_manifest)
            summary = {"index_v13": CFG.index_v13, "prefix_v13": CFG.prefix_v13,
                       "pdf_dir": str(Path(CFG.pdf_dir).resolve()), **totals}
            with open(summary_path, "w", encoding="utf-8") as f:
                json.dump(summary, f, ensure_ascii=False, indent=2)
            print("\n" + "="*50)
            print(json.dumps(summary, ensure_ascii=False, indent=2))
            print("="*50 + "\n")

    # ── Abre o browser e sobe o Flask ──────────────────────────────────────
    _url = "http://localhost:5000"
    threading.Timer(1.5, lambda: webbrowser.open(_url)).start()
    print(f"\n🚀 Servidor Flask iniciando em {_url}\n")

    flask_thread = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False),
        daemon=True
    )
    flask_thread.start()
    print("✅ Flask rodando em background — kernel continua livre")


main()

[INFO] Índice V13: idx:hyparag_v13  |  Prefixo: doc:v13:
[INFO] Pasta PDFs: C:\Users\adede\OneDrive\Documents\hyparag\documentos
[OK] Índice já existe: idx:hyparag_v13

[OK] Nenhum PDF novo ou alterado — reindexação não necessária.


🚀 Servidor Flask iniciando em http://localhost:5000

✅ Flask rodando em background — kernel continua livre


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.15.187:5000
Press CTRL+C to quit
127.0.0.1 - - [01/Mar/2026 02:48:21] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [01/Mar/2026 02:48:21] "GET /favicon.ico HTTP/1.1" 404 -


[HyPA-RAG] nivel=media | filtro=(nenhum)
[DEBUG knn_dense_filtrado] q='*=>[KNN 20 @embedding $vec AS score]' docs=20
[DEBUG sparse_text_filtrado] q='(funciona disponibilização dados públicos parte)' docs=0
Carregando cross-encoder...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 431.30it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

127.0.0.1 - - [01/Mar/2026 14:17:05] "POST /perguntar HTTP/1.1" 200 -


[HyPA-RAG] nivel=simples | filtro=(nenhum)
[DEBUG knn_dense_filtrado] q='*=>[KNN 12 @embedding $vec AS score]' docs=12
[DEBUG sparse_text_filtrado] q='(decreto decretos fala carga horária professores)' docs=0


127.0.0.1 - - [01/Mar/2026 14:22:09] "POST /perguntar HTTP/1.1" 200 -


[HyPA-RAG] nivel=simples | filtro=@area:{transparencia_controle}
[DEBUG knn_dense_filtrado] q='(@area:{transparencia_controle})=>[KNN 12 @embedding $vec AS score]' docs=12
[DEBUG sparse_text_filtrado] q='(@area:{transparencia_controle}) (finalidade sic)' docs=0


127.0.0.1 - - [01/Mar/2026 14:24:59] "POST /perguntar HTTP/1.1" 200 -


In [ ]:
# ── Geração de candidatos para o eval_set ────────────────────────────────────
# Cole numa célula nova do notebook e rode.
# Gera eval_candidatos.json com os 5 chunks mais relevantes por pergunta
# para você validar e preencher o ground_truth.

import json
import numpy as np
from redis.commands.search.query import Query   # import fora do loop

EVAL_PATH       = "eval_set_transparencia.jsonl"   # ajuste o caminho se necessário
OUT_PATH        = "eval_candidatos.json"
N_CANDIDATES    = 5
CONTENT_MAXCHAR = 800   # aumentado: 500 cortava artigos no meio

# ── Leitura do eval_set ───────────────────────────────────────────────────────
with open(EVAL_PATH, encoding="utf-8") as f:
    rows = [json.loads(l) for l in f if l.strip()]

print(f"Perguntas carregadas: {len(rows)}\n")

candidatos = []

for row in rows:
    pergunta = row["question"]
    area     = row["area"]

    # ── Embedding da pergunta ─────────────────────────────────────────────────
    emb = encode_queries([pergunta])[0]
    vec = np.array(emb, dtype=EMBED_DTYPE).tobytes()

    filtro = f"@area:{{{area}}}"

    # ── KNN denso filtrado por área ───────────────────────────────────────────
    q = (
        Query(f"({filtro})=>[KNN {N_CANDIDATES} @embedding $vec AS score]")
        .sort_by("score")
        .paging(0, N_CANDIDATES)
        .return_fields("content", "source", "chunk_id", "score")
        .dialect(2)
    )

    try:
        res = r.ft(REDIS_INDEX).search(q, query_params={"vec": vec})
        docs = getattr(res, "docs", [])
    except Exception as e:
        print(f"  [ERRO] {row['id']}: {e}")
        docs = []

    chunks = []
    for d in docs:
        raw_score  = float(getattr(d, "score", 1.0))
        cosine_sim = round(1.0 - (raw_score / 2.0), 4)
        content    = as_text(getattr(d, "content", ""))

        # Trunca no último caractere de pontuação para não cortar no meio
        if len(content) > CONTENT_MAXCHAR:
            truncated = content[:CONTENT_MAXCHAR]
            last_stop = max(
                truncated.rfind("."),
                truncated.rfind(";"),
                truncated.rfind("\n"),
            )
            content = truncated[:last_stop + 1] if last_stop > 100 else truncated

        chunks.append({
            "rank":     len(chunks) + 1,
            "chunk_id": as_text(getattr(d, "chunk_id", "")),
            "source":   as_text(getattr(d, "source",   "")),
            "score":    cosine_sim,
            "content":  content,
        })

    candidatos.append({
        "id":          row["id"],
        "area":        area,
        "complexidade": row["complexidade"],
        "tipo":        row["tipo"],
        "question":    pergunta,
        "n_chunks":    len(chunks),
        "chunks":      chunks,
        # Campos para você preencher depois de validar os chunks:
        "ground_truth_answer":   "",
        "ground_truth_contexts": [],
    })

    status = "✓" if chunks else "⚠ sem resultados"
    print(f"  {status} {row['id']} [{area}] {pergunta[:55]}")

# ── Salva com ensure_ascii=True (evita problema de encoding no Windows) ───────
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(candidatos, f, ensure_ascii=True, indent=2)

print(f"\n✅ Salvo em: {OUT_PATH}")
print(f"   Perguntas: {len(candidatos)}")
print(f"   Chunks/pergunta (média): {sum(c['n_chunks'] for c in candidatos)/len(candidatos):.1f}")

Perguntas carregadas: 23

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.15.187:5000
Press CTRL+C to quit


  ✓ t001 [transparencia_controle] O que é transparência ativa?
  ✓ t002 [transparencia_controle] O que é transparência passiva?
  ✓ t003 [transparencia_controle] Qual a diferença entre transparência ativa e transparên
  ✓ t004 [transparencia_controle] Qual é a lei de acesso à informação do estado do Pará?
  ✓ t005 [transparencia_controle] Qual é o decreto estadual de transparência pública do P
  ✓ t006 [transparencia_controle] O que estabelece a Lei Federal 12.527/2011?
  ✓ t007 [transparencia_controle] Quais informações os órgãos públicos são obrigados a di
  ✓ t008 [transparencia_controle] Qual o prazo para resposta a um pedido de acesso à info
  ✓ t009 [transparencia_controle] Como um cidadão pode fazer um pedido de acesso à inform
  ✓ t010 [transparencia_controle] O que é o e-SIC e qual sua finalidade?
  ✓ t011 [transparencia_controle] Explique como funciona a transparência pública no estad
  ✓ t012 [transparencia_controle] Quais são as hipóteses de sigilo que permitem restringi
  

In [9]:
avaliar_dataset(
    eval_path="eval_set_auto.jsonl",
    out_path="eval_results.jsonl"
)

[HyPA-RAG] nivel=media | filtro=(nenhum)
[DEBUG knn_dense_filtrado] q='*=>[KNN 20 @embedding $vec AS score]' docs=20
[DEBUG sparse_text_filtrado] q='(funciona disponibilização dados públicos parte)' docs=0


C:\Users\adede\AppData\Local\Temp\ipykernel_21888\555442526.py:1005: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


[HyPA-RAG] nivel=media | filtro=@ano:[2015 2015]
[DEBUG knn_dense_filtrado] q='(@ano:[2015 2015])=>[KNN 20 @embedding $vec AS score]' docs=20
[DEBUG sparse_text_filtrado] q='(@ano:[2015 2015]) (decreto decretos 1359 2015 está definida)' docs=0
[HyPA-RAG] nivel=simples | filtro=@area:{transparencia_controle} @ano:[2015 2015]
[DEBUG knn_dense_filtrado] q='(@area:{transparencia_controle} @ano:[2015 2015])=>[KNN 12 @embedding $vec AS score]' docs=12
[DEBUG sparse_text_filtrado] q='(@area:{transparencia_controle} @ano:[2015 2015]) (decreto decretos 1359 2015 prevê pedidos)' docs=0
[HyPA-RAG] nivel=simples | filtro=@area:{transparencia_controle}
[DEBUG knn_dense_filtrado] q='(@area:{transparencia_controle})=>[KNN 12 @embedding $vec AS score]' docs=12
[DEBUG sparse_text_filtrado] q='(@area:{transparencia_controle}) (ato normativo acesso informação estado)' docs=8
[HyPA-RAG] nivel=complexa | filtro=@area:{transparencia_controle}
[DEBUG knn_dense_filtrado] q='(@area:{transparencia_controle})=>[

In [10]:
#RESULTADO DA AVALIAÇÃO - LEITURA DO JSON EVAL_RESULTS.JSONL
import json, numpy as np

with open("eval_results.jsonl", "r", encoding="utf-8") as f:
    results = [json.loads(l) for l in f if l.strip()]

metricas = ["hit_rate", "mrr", "ndcg", "context_recall",
            "context_precision", "faithfulness", "answer_relevance", "answer_correctness"]

print(f"Total: {len(results)} perguntas\n")
print(f"{'Métrica':<25} {'Média':>8}  {'Min':>8}  {'Max':>8}")
print("-" * 55)
for k in metricas:
    vals = [r["metrics"][k] for r in results if r.get("metrics", {}).get(k) is not None]
    if vals:
        print(f"{k:<25} {np.mean(vals):>8.4f}  {min(vals):>8.4f}  {max(vals):>8.4f}")

Total: 11 perguntas

Métrica                      Média       Min       Max
-------------------------------------------------------
hit_rate                    1.0000    1.0000    1.0000
mrr                         1.0000    1.0000    1.0000
ndcg                        1.6096    1.0000    2.1309
context_recall              1.0000    1.0000    1.0000
context_precision           0.6970    0.0000    1.0000
faithfulness                0.7364    0.0000    1.0000
answer_relevance            0.8560    0.8081    0.9129
answer_correctness          0.7182    0.2000    1.0000


127.0.0.1 - - [01/Mar/2026 15:53:13] "GET / HTTP/1.1" 200 -
